In [0]:
from pyspark.sql.functions import col, year, month, avg, max, min, count
import matplotlib.pyplot as plt
import pandas as pd

base_path = "/Volumes/workspace/default/datasets"

hourly_path = f"{base_path}/all_buoys_hourly_data_new.parquet"
metadata_path = f"{base_path}/buoy_metadata_in_water.csv"

In [0]:
# Load the parquet file
df = spark.read.parquet(hourly_path)
df_metadata = spark.read.csv(metadata_path, header=True, inferSchema=True)

print("Hourly data loaded!")

In [0]:
# Check the schema
df.printSchema()

In [0]:
display(df.limit(100))

In [0]:
#Register DataFrames as Temporary Views
df.createOrReplaceTempView("buoys")
df_metadata.createOrReplaceTempView("metadata")

print("Views created!")

In [0]:
%sql

SELECT *
FROM buoys
WHERE water_temp_c is not null
LIMIT 5

In [0]:
from pyspark.sql.functions import when

df = df.withColumn("ocean_region",
    # Southern Ocean - Excluded. Dataset contains no buoys below -55° latitude.
    # Arctic Ocean
    when(col("latitude") > 66.56, "Arctic Ocean")
    # Indian Ocean
    .when(
        (col("latitude").between(-55, 31.18)) & 
        (col("longitude").between(20, 146.9)),
        "Indian Ocean"
    )
    # North Atlantic
    .when(
        (col("latitude").between(-0.936, 68.64)) & 
        (col("longitude").between(-98.05, 12)),
        "North Atlantic"
    )
    # South Atlantic
    .when(
        (col("latitude").between(-55, 0.075)) & 
        (col("longitude").between(-69.6, 20)),
        "South Atlantic"
    )
    # North Pacific - wraps around 180° line
    .when(
        (col("latitude").between(0, 66.56)) & 
        ((col("longitude") >= 117.5) | (col("longitude") <= -76.98)),
        "North Pacific"
    )
    # South Pacific - wraps around 180° line
    .when(
        (col("latitude").between(-55, 3.4)) & 
        ((col("longitude") >= 130.11) | (col("longitude") <= -67.27)),
        "South Pacific"
    )
    .otherwise("Unknown")
)

print("Ocean region column added!")

In [0]:
display(
    df.groupBy("ocean_region")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
df.createOrReplaceTempView("buoys")
print("View updated with ocean regions!")

In [0]:
%sql
SELECT 
    ocean_region,
    min(latitude) as min_lat,
    max(latitude) as max_lat,
    min(longitude) as min_lon,
    max(longitude) as max_lon
FROM buoys
GROUP BY ocean_region
ORDER BY ocean_region

In [0]:
from pyspark.sql.functions import year, round

df_temp_trend = df.filter(col("water_temp_c").isNotNull()) \
    .groupBy(year("datetime_utc").alias("year")) \
    .agg(round(avg("water_temp_c"), 4).alias("avg_water_temp")) \
    .orderBy("year")

display(df_temp_trend)

In [0]:
# Convert to pandas for plotting
temp_trend_pd = df_temp_trend.toPandas()

plt.figure(figsize=(14, 6))
plt.plot(temp_trend_pd["year"], temp_trend_pd["avg_water_temp"], 
         color="steelblue", linewidth=2, marker="o", markersize=4)

# Add a trend line
import numpy as np
z = np.polyfit(temp_trend_pd["year"], temp_trend_pd["avg_water_temp"], 1)
p = np.poly1d(z)
plt.plot(temp_trend_pd["year"], p(temp_trend_pd["year"]), 
         color="red", linewidth=1.5, linestyle="--", label="Trend")

plt.xlabel("Year")
plt.ylabel("Average Sea Surface Temperature (°C)")
plt.title("Global Average Sea Surface Temperature (1980-2024)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import year, countDistinct

df_buoy_coverage = df.groupBy(year("datetime_utc").alias("year")) \
    .agg(countDistinct("buoy_id").alias("active_buoys")) \
    .orderBy("year")

display(df_buoy_coverage)

In [0]:
# Convert to pandas for plotting
buoy_coverage_pd = df_buoy_coverage.toPandas()

plt.figure(figsize=(14, 6))
plt.plot(buoy_coverage_pd["year"], buoy_coverage_pd["active_buoys"],
         color="steelblue", linewidth=2, marker="o", markersize=4)

plt.xlabel("Year")
plt.ylabel("Number of Active Buoys")
plt.title("Number of Active Buoys by Year (1999-2025)")
plt.axvline(x=2008, color="red", linewidth=1.5,
            linestyle="--", label="Coverage becomes reliable (~50 buoys)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# A rogue wave is traditionally defined as a wave more than twice
# the significant wave height of surrounding seas. However since we
# don't have surrounding sea context, we'll use a statistical approach
# instead. Any wave height more than 3 standard deviations above
# the mean is considered anomalously large.

from pyspark.sql.functions import stddev, mean

# Calculate the mean and standard deviation of wave height
wave_stats = df.filter(col("wave_height_m").isNotNull()) \
    .agg(
        mean("wave_height_m").alias("mean_wave"),
        stddev("wave_height_m").alias("stddev_wave")
    ).collect()[0]

mean_wave = wave_stats["mean_wave"]
stddev_wave = wave_stats["stddev_wave"]
threshold = mean_wave + (3 * stddev_wave)

print(f"Mean wave height:      {mean_wave:.4f}m")
print(f"Std deviation:         {stddev_wave:.4f}m")
print(f"Rogue wave threshold:  {threshold:.4f}m")

In [0]:
df_rogue = df.filter(
    (col("wave_height_m").isNotNull()) &
    (col("wave_height_m") >= threshold)
).select(
    "buoy_id",
    "datetime_utc",
    "wave_height_m",
    "water_temp_c",
    "wind_speed_ms",
    "latitude",
    "longitude",
    "ocean_region"
).orderBy(col("wave_height_m").desc())

print(f"Total rogue wave events: {df_rogue.count()}")
display(df_rogue.limit(25))

In [0]:
df_rogue_by_region = df_rogue.groupBy("ocean_region") \
    .agg(
        count("wave_height_m").alias("rogue_wave_count"),
        round(avg("wave_height_m"), 2).alias("avg_rogue_height"),
        round(max("wave_height_m"), 2).alias("max_rogue_height")
    ).orderBy(col("rogue_wave_count").desc())

display(df_rogue_by_region)

In [0]:
rogue_region_pd = df_rogue_by_region.toPandas()

plt.figure(figsize=(12, 6))
bars = plt.bar(rogue_region_pd["ocean_region"], 
               rogue_region_pd["rogue_wave_count"],
               color="steelblue")

# Add value labels on top of each bar
for bar, val in zip(bars, rogue_region_pd["rogue_wave_count"]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f"{val:,}", ha="center", va="bottom", fontsize=10)

plt.xlabel("Ocean Region")
plt.ylabel("Number of Rogue Wave Events")
plt.title("Rogue Wave Events by Ocean Region")
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()